# Enhanced Sales Intelligence Framework
## Quantitative NLP-Based Analysis for Sales Transcripts

This notebook combines statistical analysis with quantitative NLP methods to extract deep business intelligence from sales conversations. No GenAI - only measurable, numerical NLP outputs.

In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import re
from datetime import datetime, timedelta
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')

# NLP Libraries for Quantitative Analysis
try:
    from textblob import TextBlob
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    import spacy
    from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    import nltk
    from nltk.corpus import stopwords
    from nltk.tokenize import word_tokenize, sent_tokenize
    from nltk.tag import pos_tag
    from nltk.chunk import ne_chunk
    
    # Download required NLTK data
    nltk.download('punkt', quiet=True)
    nltk.download('vader_lexicon', quiet=True)
    nltk.download('averaged_perceptron_tagger', quiet=True)
    nltk.download('maxent_ne_chunker', quiet=True)
    nltk.download('words', quiet=True)
    nltk.download('stopwords', quiet=True)
    
    NLP_AVAILABLE = True
    print("✓ NLP libraries loaded successfully")
except ImportError as e:
    NLP_AVAILABLE = False
    print(f"⚠ NLP libraries not available: {e}")
    print("Install with: pip install textblob vaderSentiment spacy scikit-learn nltk")

# Set styling
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Enhanced Sales Intelligence Framework Loaded")

⚠ NLP libraries not available: No module named 'textblob'
Install with: pip install textblob vaderSentiment spacy scikit-learn nltk
Enhanced Sales Intelligence Framework Loaded


## 1. Enhanced Transcript Parser with NLP

In [34]:
class EnhancedTranscriptParser:
    """Enhanced parser with NLP preprocessing"""
    
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None
        self.vader_analyzer = SentimentIntensityAnalyzer() if NLP_AVAILABLE else None
    
    def time_to_seconds(self, time_str):
        """Convert timestamp to seconds"""
        time_parts = time_str.replace('.', ':').split(':')
        if len(time_parts) == 4:
            hours, minutes, seconds, milliseconds = map(int, time_parts)
            return hours * 3600 + minutes * 60 + seconds + milliseconds / 1000
        return 0
    
    def extract_speaker(self, text):
        """Extract speaker from text using patterns"""
        patterns = [
            r'^([^:()]+)\s*\([^)]+\)\s*:', # Name (Role):
            r'^([^:]+):', # Name:
            r'^\[([^\]]+)\]', # [Name]
            r'^([A-Z][a-z]+)\s*-', # Name -
        ]
        
        for pattern in patterns:
            match = re.match(pattern, text.strip())
            if match:
                speaker = match.group(1).strip()
                message = text[match.end():].strip()
                return speaker, message
        
        return "Unknown", text
    
    def analyze_text_with_nlp(self, text):
        """Extract NLP features from text"""
        if not NLP_AVAILABLE or not text.strip():
            return {}
        
        features = {}
        
        # Sentiment Analysis
        try:
            # VADER Sentiment (great for conversational text)
            vader_scores = self.vader_analyzer.polarity_scores(text)
            features['sentiment_positive'] = vader_scores['pos']
            features['sentiment_negative'] = vader_scores['neg']
            features['sentiment_neutral'] = vader_scores['neu']
            features['sentiment_compound'] = vader_scores['compound']
            
            # TextBlob Sentiment
            blob = TextBlob(text)
            features['polarity'] = blob.sentiment.polarity
            features['subjectivity'] = blob.sentiment.subjectivity
        except Exception as e:
            features.update({
                'sentiment_positive': 0, 'sentiment_negative': 0,
                'sentiment_neutral': 1, 'sentiment_compound': 0,
                'polarity': 0, 'subjectivity': 0
            })
        
        # Linguistic Features
        try:
            tokens = word_tokenize(text.lower())
            features['sentence_count'] = len(sent_tokenize(text))
            features['avg_sentence_length'] = len(tokens) / max(features['sentence_count'], 1)
            
            # POS tagging for linguistic analysis
            pos_tags = pos_tag(tokens)
            pos_counts = Counter([tag for word, tag in pos_tags])
            
            features['noun_count'] = pos_counts.get('NN', 0) + pos_counts.get('NNS', 0)
            features['verb_count'] = pos_counts.get('VB', 0) + pos_counts.get('VBG', 0)
            features['adjective_count'] = pos_counts.get('JJ', 0)
            features['modal_count'] = pos_counts.get('MD', 0)  # will, can, should, etc.
            
        except Exception as e:
            features.update({
                'sentence_count': 1, 'avg_sentence_length': len(text.split()),
                'noun_count': 0, 'verb_count': 0, 'adjective_count': 0, 'modal_count': 0
            })
        
        return features
    
    def parse(self):
        """Parse SRT file with enhanced NLP analysis"""
        with open(self.file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        
        blocks = content.strip().split('\n\n')
        conversations = []
        
        for block in blocks:
            lines = block.strip().split('\n')
            if len(lines) >= 3:
                try:
                    subtitle_num = int(lines[0])
                    timestamp_line = lines[1]
                    
                    # Fixed regex pattern for timestamp extraction
                    time_match = re.match(r'(\d{2}:\d{2}:\d{2}[,.]\d{3}) --> (\d{2}:\d{2}:\d{2}[,.]\d{3})', timestamp_line)
                    if time_match:
                        start_time = time_match.group(1).replace(',', '.')
                        end_time = time_match.group(2).replace(',', '.')
                        
                        text = ' '.join(lines[2:])
                        speaker, message = self.extract_speaker(text)
                        
                        # Basic features
                        conv_data = {
                            'subtitle_num': subtitle_num,
                            'start_time': start_time,
                            'end_time': end_time,
                            'speaker': speaker,
                            'message': message,
                            'full_text': text
                        }
                        
                        # Add NLP features
                        nlp_features = self.analyze_text_with_nlp(message)
                        conv_data.update(nlp_features)
                        
                        conversations.append(conv_data)
                        
                except (ValueError, IndexError) as e:
                    continue
        
        self.df = pd.DataFrame(conversations)
        if not self.df.empty:
            # Basic metrics
            self.df['start_seconds'] = self.df['start_time'].apply(self.time_to_seconds)
            self.df['end_seconds'] = self.df['end_time'].apply(self.time_to_seconds)
            self.df['duration'] = self.df['end_seconds'] - self.df['start_seconds']
            self.df['word_count'] = self.df['message'].apply(lambda x: len(str(x).split()))
            self.df['char_count'] = self.df['message'].apply(lambda x: len(str(x)))
            self.df['question_marks'] = self.df['message'].apply(lambda x: str(x).count('?'))
            self.df['exclamation_marks'] = self.df['message'].apply(lambda x: str(x).count('!'))
            
        return self.df

# Load and parse transcript with NLP enhancement
parser = EnhancedTranscriptParser('transcript.srt')
df = parser.parse()

print(f"✓ Parsed {len(df)} conversation segments with NLP features")
print(f"✓ Unique speakers: {df['speaker'].nunique()}")
print(f"✓ Total duration: {df['end_seconds'].max()/60:.1f} minutes")

# Show available columns
print(f"✓ Available columns: {', '.join(df.columns.tolist())}")

# Check what NLP features are available and show sample
available_nlp_columns = [col for col in df.columns if col.startswith(('sentiment_', 'polarity', 'subjectivity'))]
if available_nlp_columns:
    print(f"✓ NLP features extracted: {len(available_nlp_columns)} sentiment metrics")
    sample_columns = ['speaker', 'message'] + available_nlp_columns[:3]
    display(df[sample_columns].head())
else:
    print("⚠ NLP libraries not available - using basic analysis only")
    if NLP_AVAILABLE:
        print("✓ NLP libraries loaded but no sentiment features found")
    else:
        print("💡 Install NLP libraries with: pip install textblob vaderSentiment spacy scikit-learn nltk")
    
    # Show basic analysis columns
    basic_columns = ['speaker', 'message', 'word_count', 'question_marks', 'duration']
    display(df[basic_columns].head())

✓ Parsed 67 conversation segments with NLP features
✓ Unique speakers: 3
✓ Total duration: 25.3 minutes
✓ Available columns: subtitle_num, start_time, end_time, speaker, message, full_text, start_seconds, end_seconds, duration, word_count, char_count, question_marks, exclamation_marks
⚠ NLP libraries not available - using basic analysis only
💡 Install NLP libraries with: pip install textblob vaderSentiment spacy scikit-learn nltk


,speaker,message,word_count,question_marks,duration
0,Arjun,"Hi Meera, good afternoon. Can you hear me clea...",9,1,18.0
1,Meera,"Yes, loud and clear. Thanks for joining. I kno...",14,0,9.5
2,Arjun,"Absolutely. I always say, the hardest part of ...",24,1,13.5
3,Meera,Busy but good. We’re preparing for a board upd...,23,0,14.5
4,Arjun,Perfect. Then let’s make this session useful. ...,38,1,15.5


## 2. Financial Intelligence Extractor

In [35]:
class FinancialIntelligenceExtractor:
    """Extract financial intelligence using NLP and pattern recognition"""
    
    def __init__(self, df):
        self.df = df.copy()
        self.financial_patterns = {
            'currency': r'\$[\d,]+(?:\.\d{2})?|USD\s*[\d,]+|\d+(?:\.\d+)?\s*(?:dollars?|million|billion|k|thousand)',
            'budget': r'budget|cost|price|pricing|expensive|cheap|afford|invest|spend|ROI|revenue',
            'timeline_financial': r'\d+\s*(?:months?|weeks?|quarters?)\s*(?:budget|cost|investment)',
            'approval_process': r'board|approve|authorization|procurement|RFP|budget\s*approval'
        }
        
        self.cost_sensitivity_keywords = {
            'high_sensitivity': ['expensive', 'costly', 'budget constraint', 'tight budget', 'can\'t afford'],
            'medium_sensitivity': ['reasonable', 'fair price', 'cost-effective', 'value'],
            'low_sensitivity': ['willing to invest', 'budget available', 'price not an issue', 'worth it']
        }
        
        self.roi_indicators = ['ROI', 'return on investment', 'payback', 'cost savings', 'efficiency gains', 'revenue impact']
    
    def extract_financial_mentions(self, text):
        """Extract financial amounts and context"""
        text_lower = text.lower()
        
        # Extract currency amounts
        currency_matches = re.findall(self.financial_patterns['currency'], text, re.IGNORECASE)
        
        # Convert to numerical values where possible
        amounts = []
        for match in currency_matches:
            try:
                # Extract numerical part
                num_match = re.search(r'[\d,]+(?:\.\d+)?', match)
                if num_match:
                    num_str = num_match.group().replace(',', '')
                    num_val = float(num_str)
                    
                    # Apply multipliers
                    if 'k' in match.lower() or 'thousand' in match.lower():
                        num_val *= 1000
                    elif 'million' in match.lower():
                        num_val *= 1000000
                    elif 'billion' in match.lower():
                        num_val *= 1000000000
                    
                    amounts.append(num_val)
            except:
                continue
        
        return {
            'currency_mentions': len(currency_matches),
            'amounts_found': amounts,
            'max_amount': max(amounts) if amounts else 0,
            'total_amount': sum(amounts) if amounts else 0
        }
    
    def calculate_cost_sensitivity(self, text):
        """Calculate cost sensitivity score"""
        text_lower = text.lower()
        
        sensitivity_score = 0
        sensitivity_indicators = 0
        
        for level, keywords in self.cost_sensitivity_keywords.items():
            for keyword in keywords:
                if keyword in text_lower:
                    sensitivity_indicators += 1
                    if level == 'high_sensitivity':
                        sensitivity_score += 3
                    elif level == 'medium_sensitivity':
                        sensitivity_score += 2
                    else:  # low_sensitivity
                        sensitivity_score += 1
        
        return {
            'cost_sensitivity_score': sensitivity_score,
            'cost_sensitivity_indicators': sensitivity_indicators,
            'cost_sensitivity_level': 'High' if sensitivity_score >= 6 else 'Medium' if sensitivity_score >= 3 else 'Low'
        }
    
    def analyze_roi_discussion(self, text):
        """Analyze ROI and value discussion"""
        text_lower = text.lower()
        
        roi_mentions = sum(1 for indicator in self.roi_indicators if indicator.lower() in text_lower)
        
        # Look for specific ROI patterns
        roi_patterns = [
            r'\d+%\s*ROI',
            r'ROI\s*of\s*\d+',
            r'payback\s*in\s*\d+\s*(?:months?|years?)',
            r'save\s*\$[\d,]+',
            r'reduce\s*cost.*\d+%'
        ]
        
        quantified_roi = sum(1 for pattern in roi_patterns if re.search(pattern, text_lower))
        
        return {
            'roi_mentions': roi_mentions,
            'quantified_roi': quantified_roi,
            'roi_focus': roi_mentions > 0 or quantified_roi > 0
        }
    
    def extract_approval_process_info(self, text):
        """Extract information about approval processes"""
        text_lower = text.lower()
        
        approval_indicators = {
            'board_approval': any(word in text_lower for word in ['board', 'board approval', 'board meeting']),
            'procurement_process': any(word in text_lower for word in ['procurement', 'rfp', 'rfq', 'tender']),
            'budget_approval': any(word in text_lower for word in ['budget approval', 'budget committee', 'finance team']),
            'direct_purchase': any(phrase in text_lower for phrase in ['direct purchase', 'can buy directly', 'no approval needed'])
        }
        
        complexity_score = sum(1 for key, value in approval_indicators.items() if value and key != 'direct_purchase')
        if approval_indicators['direct_purchase']:
            complexity_score = max(0, complexity_score - 2)
        
        return {
            **approval_indicators,
            'approval_complexity_score': complexity_score,
            'approval_complexity_level': 'High' if complexity_score >= 2 else 'Medium' if complexity_score == 1 else 'Low'
        }
    
    def analyze_conversation(self):
        """Analyze entire conversation for financial intelligence"""
        financial_data = []
        
        for idx, row in self.df.iterrows():
            text = row['message']
            
            # Extract various financial features
            financial_mentions = self.extract_financial_mentions(text)
            cost_sensitivity = self.calculate_cost_sensitivity(text)
            roi_analysis = self.analyze_roi_discussion(text)
            approval_info = self.extract_approval_process_info(text)
            
            # Combine all features
            segment_data = {
                'segment_id': idx,
                'speaker': row['speaker'],
                'message': text,
                **financial_mentions,
                **cost_sensitivity,
                **roi_analysis,
                **approval_info
            }
            
            financial_data.append(segment_data)
        
        return pd.DataFrame(financial_data)
    
    def generate_financial_summary(self):
        """Generate overall financial intelligence summary"""
        financial_df = self.analyze_conversation()
        
        # Overall metrics
        total_currency_mentions = financial_df['currency_mentions'].sum()
        max_deal_size = financial_df['max_amount'].max()
        avg_cost_sensitivity = financial_df['cost_sensitivity_score'].mean()
        roi_discussions = financial_df['roi_mentions'].sum()
        
        # By speaker analysis
        speaker_financial = financial_df.groupby('speaker').agg({
            'currency_mentions': 'sum',
            'cost_sensitivity_score': 'mean',
            'roi_mentions': 'sum',
            'approval_complexity_score': 'mean'
        }).round(2)
        
        # Approval process complexity
        approval_complexity = financial_df[['board_approval', 'procurement_process', 'budget_approval', 'direct_purchase']].any()
        
        summary = {
            'total_currency_mentions': total_currency_mentions,
            'estimated_deal_size': max_deal_size,
            'average_cost_sensitivity': avg_cost_sensitivity,
            'roi_discussion_segments': roi_discussions,
            'approval_requirements': approval_complexity.to_dict(),
            'speaker_analysis': speaker_financial.to_dict(),
            'financial_segments': len(financial_df[financial_df['currency_mentions'] > 0])
        }
        
        return summary, financial_df

# Extract financial intelligence
financial_extractor = FinancialIntelligenceExtractor(df)
financial_summary, financial_df = financial_extractor.generate_financial_summary()

print("\n🎯 FINANCIAL INTELLIGENCE SUMMARY:")
print(f"💰 Currency mentions: {financial_summary['total_currency_mentions']}")
print(f"📊 Estimated deal size: ${financial_summary['estimated_deal_size']:,.0f}" if financial_summary['estimated_deal_size'] > 0 else "📊 No specific deal size mentioned")
print(f"🎚️ Average cost sensitivity: {financial_summary['average_cost_sensitivity']:.2f}/10")
print(f"💡 ROI discussion segments: {financial_summary['roi_discussion_segments']}")
print(f"🔐 Financial segments found: {financial_summary['financial_segments']}")

# Show financial analysis by speaker
print("\nSPEAKER FINANCIAL ANALYSIS:")
for speaker, data in financial_summary['speaker_analysis']['currency_mentions'].items():
    print(f"{speaker}: {data} currency mentions, {financial_summary['speaker_analysis']['cost_sensitivity_score'][speaker]:.2f} cost sensitivity")


🎯 FINANCIAL INTELLIGENCE SUMMARY:
💰 Currency mentions: 3
📊 Estimated deal size: $100,000
🎚️ Average cost sensitivity: 0.09/10
💡 ROI discussion segments: 2
🔐 Financial segments found: 3

SPEAKER FINANCIAL ANALYSIS:
Arjun: 2 currency mentions, 0.06 cost sensitivity
Meera: 1 currency mentions, 0.12 cost sensitivity
Small talk as both smile and exchange pleasantries before leaving the call.: 0 currency mentions, 0.00 cost sensitivity


## 3. Competitive Intelligence Analyzer

In [36]:
class CompetitiveIntelligenceAnalyzer:
    """Analyze competitive mentions and positioning using NLP"""
    
    def __init__(self, df):
        self.df = df.copy()
        
        # Common competitor patterns (can be customized)
        self.competitor_patterns = [
            # Technology companies
            r'\b(?:AWS|Amazon|Microsoft|Azure|Google|GCP|Oracle|SAP|Salesforce|IBM)\b',
            # Generic competitor indicators
            r'\b(?:competitor|competition|rival|alternative|other\s+(?:vendor|provider|solution))s?\b',
            # Specific mentions in the transcript
            r'\b(?:Project44|FourKites|Flexport)\b',
        ]
        
        # Competitive sentiment indicators
        self.competitive_sentiment = {
            'positive_competitor': ['leader', 'dominant', 'established', 'reliable', 'stable', 'proven'],
            'negative_competitor': ['expensive', 'slow', 'rigid', 'complex', 'outdated', 'inflexible'],
            'opportunity_indicators': ['gap', 'missing', 'lack', 'doesn\'t have', 'weakness', 'problem with']
        }
        
        # Differentiation keywords
        self.differentiation_indicators = {
            'speed': ['faster', 'quicker', 'rapid', 'quick deployment', 'fast implementation'],
            'cost': ['cheaper', 'cost-effective', 'affordable', 'lower cost', 'budget-friendly'],
            'flexibility': ['flexible', 'customizable', 'adaptable', 'configurable', 'tailored'],
            'innovation': ['AI-driven', 'advanced', 'cutting-edge', 'innovative', 'next-generation'],
            'service': ['better support', 'personalized', 'dedicated', 'responsive']
        }
    
    def extract_competitor_mentions(self, text):
        """Extract competitor mentions and context"""
        competitor_data = {
            'competitor_mentions': 0,
            'competitors_found': [],
            'competitor_context': []
        }
        
        text_lower = text.lower()
        
        for pattern in self.competitor_patterns:
            matches = re.finditer(pattern, text, re.IGNORECASE)
            for match in matches:
                competitor_data['competitor_mentions'] += 1
                competitor_name = match.group()
                competitor_data['competitors_found'].append(competitor_name)
                
                # Extract context (surrounding words)
                start = max(0, match.start() - 100)
                end = min(len(text), match.end() + 100)
                context = text[start:end].strip()
                competitor_data['competitor_context'].append(context)
        
        return competitor_data
    
    def analyze_competitive_sentiment(self, text, competitor_mentions):
        """Analyze sentiment around competitor mentions"""
        if competitor_mentions == 0:
            return {
                'competitor_positive_sentiment': 0,
                'competitor_negative_sentiment': 0,
                'opportunity_indicators': 0,
                'competitive_sentiment_score': 0
            }
        
        text_lower = text.lower()
        
        positive_count = sum(1 for keyword in self.competitive_sentiment['positive_competitor'] 
                           if keyword in text_lower)
        negative_count = sum(1 for keyword in self.competitive_sentiment['negative_competitor'] 
                           if keyword in text_lower)
        opportunity_count = sum(1 for keyword in self.competitive_sentiment['opportunity_indicators'] 
                              if keyword in text_lower)
        
        # Calculate competitive sentiment score (-1 to 1)
        # Negative = good for us (competitor weakness), Positive = threat (competitor strength)
        total_mentions = positive_count + negative_count + opportunity_count
        if total_mentions > 0:
            sentiment_score = (negative_count + opportunity_count - positive_count) / total_mentions
        else:
            sentiment_score = 0
        
        return {
            'competitor_positive_sentiment': positive_count,
            'competitor_negative_sentiment': negative_count,
            'opportunity_indicators': opportunity_count,
            'competitive_sentiment_score': round(sentiment_score, 3)
        }
    
    def analyze_differentiation_strategy(self, text):
        """Analyze differentiation messaging"""
        text_lower = text.lower()
        
        differentiation_scores = {}
        total_differentiation = 0
        
        for category, keywords in self.differentiation_indicators.items():
            score = sum(1 for keyword in keywords if keyword in text_lower)
            differentiation_scores[f'differentiation_{category}'] = score
            total_differentiation += score
        
        differentiation_scores['total_differentiation_indicators'] = total_differentiation
        
        # Determine primary differentiation strategy
        if total_differentiation > 0:
            primary_diff = max([(k.replace('differentiation_', ''), v) 
                              for k, v in differentiation_scores.items() 
                              if k.startswith('differentiation_') and v > 0], 
                             key=lambda x: x[1], default=('none', 0))
            differentiation_scores['primary_differentiation'] = primary_diff[0]
        else:
            differentiation_scores['primary_differentiation'] = 'none'
        
        return differentiation_scores
    
    def analyze_market_positioning(self, text):
        """Analyze market positioning statements"""
        text_lower = text.lower()
        
        positioning_indicators = {
            'market_leader_claims': len(re.findall(r'\b(?:leader|leading|#1|number one|top player)\b', text_lower)),
            'market_challenger_claims': len(re.findall(r'\b(?:alternative|better|superior|outperform)\b', text_lower)),
            'niche_positioning': len(re.findall(r'\b(?:specialized|focused|niche|specific to|designed for)\b', text_lower)),
            'enterprise_positioning': len(re.findall(r'\b(?:enterprise|large|scale|global|fortune)\b', text_lower)),
            'smb_positioning': len(re.findall(r'\b(?:small|medium|startup|mid-market|SMB|SME)\b', text_lower))
        }
        
        # Determine primary positioning
        primary_positioning = max(positioning_indicators.items(), key=lambda x: x[1])
        positioning_indicators['primary_positioning'] = primary_positioning[0] if primary_positioning[1] > 0 else 'undefined'
        
        return positioning_indicators
    
    def analyze_conversation(self):
        """Analyze entire conversation for competitive intelligence"""
        competitive_data = []
        
        for idx, row in self.df.iterrows():
            text = row['message']
            
            # Extract competitive features
            competitor_mentions = self.extract_competitor_mentions(text)
            competitive_sentiment = self.analyze_competitive_sentiment(text, competitor_mentions['competitor_mentions'])
            differentiation = self.analyze_differentiation_strategy(text)
            positioning = self.analyze_market_positioning(text)
            
            # Combine all features
            segment_data = {
                'segment_id': idx,
                'speaker': row['speaker'],
                'message': text,
                **competitor_mentions,
                **competitive_sentiment,
                **differentiation,
                **positioning
            }
            
            competitive_data.append(segment_data)
        
        return pd.DataFrame(competitive_data)
    
    def generate_competitive_summary(self):
        """Generate competitive intelligence summary"""
        competitive_df = self.analyze_conversation()
        
        # Extract all competitors mentioned
        all_competitors = []
        for competitors_list in competitive_df['competitors_found']:
            all_competitors.extend(competitors_list)
        
        competitor_counts = Counter(all_competitors)
        
        # Overall competitive metrics
        total_competitor_mentions = competitive_df['competitor_mentions'].sum()
        avg_competitive_sentiment = competitive_df['competitive_sentiment_score'].mean()
        total_differentiation = competitive_df['total_differentiation_indicators'].sum()
        
        # Primary differentiation strategy
        diff_strategies = competitive_df['primary_differentiation'].value_counts().to_dict()
        primary_diff = max(diff_strategies.items(), key=lambda x: x[1]) if diff_strategies else ('none', 0)
        
        # Positioning analysis
        positioning_strategies = competitive_df['primary_positioning'].value_counts().to_dict()
        primary_positioning = max(positioning_strategies.items(), key=lambda x: x[1]) if positioning_strategies else ('undefined', 0)
        
        summary = {
            'total_competitor_mentions': total_competitor_mentions,
            'unique_competitors': len(competitor_counts),
            'top_competitors': dict(competitor_counts.most_common(5)),
            'average_competitive_sentiment': round(avg_competitive_sentiment, 3),
            'total_differentiation_indicators': total_differentiation,
            'primary_differentiation_strategy': primary_diff[0],
            'primary_market_positioning': primary_positioning[0],
            'competitive_segments': len(competitive_df[competitive_df['competitor_mentions'] > 0])
        }
        
        return summary, competitive_df

# Extract competitive intelligence
competitive_analyzer = CompetitiveIntelligenceAnalyzer(df)
competitive_summary, competitive_df = competitive_analyzer.generate_competitive_summary()

print("\n🎯 COMPETITIVE INTELLIGENCE SUMMARY:")
print(f"🏢 Competitor mentions: {competitive_summary['total_competitor_mentions']}")
print(f"🌐 Unique competitors: {competitive_summary['unique_competitors']}")
print(f"🔝 Top competitors: {competitive_summary['top_competitors']}")
print(f"😊 Competitive sentiment: {competitive_summary['average_competitive_sentiment']:.3f} (-1=threat, +1=opportunity)")
print(f"🚀 Primary differentiation: {competitive_summary['primary_differentiation_strategy']}")
print(f"📍 Market positioning: {competitive_summary['primary_market_positioning']}")
print(f"📊 Competitive segments: {competitive_summary['competitive_segments']}")


🎯 COMPETITIVE INTELLIGENCE SUMMARY:
🏢 Competitor mentions: 11
🌐 Unique competitors: 6
🔝 Top competitors: {'Project44': 3, 'AWS': 2, 'competitors': 2, 'SAP': 2, 'FourKites': 1}
😊 Competitive sentiment: 0.000 (-1=threat, +1=opportunity)
🚀 Primary differentiation: none
📍 Market positioning: undefined
📊 Competitive segments: 8


## 4. Engagement Metrics Calculator with NLP

In [37]:
class EngagementMetricsCalculator:
    """Calculate engagement and buying signals using NLP"""
    
    def __init__(self, df):
        self.df = df.copy()
        
        # Buying signal indicators
        self.buying_signals = {
            'high_interest': [
                'exactly what we need', 'perfect', 'that sounds great', 'very interested',
                'when can we start', 'how soon', 'next steps', 'moving forward'
            ],
            'medium_interest': [
                'interesting', 'good point', 'makes sense', 'that works',
                'tell me more', 'how does that work', 'what would that look like'
            ],
            'agreement_indicators': [
                'yes', 'absolutely', 'definitely', 'exactly', 'right', 'correct',
                'agree', 'true', 'that\'s right', 'you\'re right'
            ]
        }
        
        # Objection and resistance indicators
        self.objection_signals = {
            'direct_objections': [
                'but', 'however', 'though', 'concern', 'worry', 'problem',
                'issue', 'challenge', 'difficult', 'not sure'
            ],
            'skepticism': [
                'really?', 'are you sure', 'doubt', 'skeptical', 'hard to believe',
                'sounds too good', 'past experience', 'last time'
            ],
            'stalling': [
                'need to think', 'discuss with team', 'review internally',
                'get back to you', 'not ready', 'too early'
            ]
        }
        
        # Urgency indicators
        self.urgency_indicators = [
            'urgent', 'asap', 'immediately', 'right away', 'as soon as possible',
            'deadline', 'time pressure', 'need it now', 'can\'t wait'
        ]
        
        # Authority indicators
        self.authority_indicators = {
            'high_authority': ['I decide', 'my decision', 'I can approve', 'my budget', 'I\'m responsible'],
            'medium_authority': ['I recommend', 'I influence', 'part of decision', 'involved in'],
            'low_authority': ['need approval', 'boss decides', 'not my decision', 'have to ask']
        }
    
    def calculate_buying_signals(self, text, speaker):
        """Calculate buying signal strength"""
        text_lower = text.lower()
        
        signal_scores = {
            'high_interest_signals': 0,
            'medium_interest_signals': 0,
            'agreement_signals': 0,
            'total_buying_signals': 0
        }
        
        # Count different types of buying signals
        for category, signals in self.buying_signals.items():
            count = sum(1 for signal in signals if signal in text_lower)
            if category == 'high_interest':
                signal_scores['high_interest_signals'] = count
                signal_scores['total_buying_signals'] += count * 3  # Weight high interest more
            elif category == 'medium_interest':
                signal_scores['medium_interest_signals'] = count
                signal_scores['total_buying_signals'] += count * 2
            elif category == 'agreement_indicators':
                signal_scores['agreement_signals'] = count
                signal_scores['total_buying_signals'] += count * 1
        
        # Calculate buying signal intensity
        word_count = len(text.split())
        if word_count > 0:
            signal_scores['buying_signal_intensity'] = signal_scores['total_buying_signals'] / word_count
        else:
            signal_scores['buying_signal_intensity'] = 0
        
        return signal_scores
    
    def calculate_objection_resistance(self, text):
        """Calculate objection and resistance levels"""
        text_lower = text.lower()
        
        resistance_scores = {
            'direct_objections': 0,
            'skepticism_indicators': 0,
            'stalling_indicators': 0,
            'total_resistance': 0
        }
        
        # Count different types of objections
        for category, objections in self.objection_signals.items():
            count = sum(1 for objection in objections if objection in text_lower)
            if category == 'direct_objections':
                resistance_scores['direct_objections'] = count
                resistance_scores['total_resistance'] += count * 3
            elif category == 'skepticism':
                resistance_scores['skepticism_indicators'] = count
                resistance_scores['total_resistance'] += count * 2
            elif category == 'stalling':
                resistance_scores['stalling_indicators'] = count
                resistance_scores['total_resistance'] += count * 2
        
        return resistance_scores
    
    def calculate_urgency_authority(self, text, speaker):
        """Calculate urgency and authority levels"""
        text_lower = text.lower()
        
        # Urgency calculation
        urgency_count = sum(1 for indicator in self.urgency_indicators if indicator in text_lower)
        
        # Authority calculation
        authority_score = 0
        authority_indicators = 0
        
        for level, indicators in self.authority_indicators.items():
            for indicator in indicators:
                if indicator in text_lower:
                    authority_indicators += 1
                    if level == 'high_authority':
                        authority_score += 3
                    elif level == 'medium_authority':
                        authority_score += 2
                    else:  # low_authority
                        authority_score += 1
        
        return {
            'urgency_indicators': urgency_count,
            'authority_score': authority_score,
            'authority_indicators': authority_indicators,
            'authority_level': 'High' if authority_score >= 6 else 'Medium' if authority_score >= 3 else 'Low'
        }
    
    def analyze_question_patterns(self, text, speaker):
        """Analyze question patterns for engagement"""
        # Different types of questions indicate different engagement levels
        question_patterns = {
            'detail_questions': r'how (?:does|do|would|can|will)\b|what (?:is|are|would|about)\b|tell me (?:more|about)\b',
            'implementation_questions': r'when (?:can|would|will)\b|how long\b|what.*timeline\b|implementation\b',
            'pricing_questions': r'how much\b|what.*cost\b|price\b|pricing\b|budget\b',
            'comparison_questions': r'(?:compared to|vs|versus|difference between|how.*different)\b',
            'technical_questions': r'(?:technical|architecture|integration|API|security)\b.*\?'
        }
        
        question_analysis = {
            'total_questions': text.count('?'),
        }
        
        for question_type, pattern in question_patterns.items():
            matches = len(re.findall(pattern, text.lower()))
            question_analysis[question_type] = matches
        
        # Calculate question engagement score
        engagement_questions = (
            question_analysis['detail_questions'] * 2 +
            question_analysis['implementation_questions'] * 3 +
            question_analysis['pricing_questions'] * 3 +
            question_analysis['technical_questions'] * 2
        )
        
        question_analysis['question_engagement_score'] = engagement_questions
        
        return question_analysis
    
    def analyze_conversation(self):
        """Analyze entire conversation for engagement metrics"""
        engagement_data = []
        
        for idx, row in self.df.iterrows():
            text = row['message']
            speaker = row['speaker']
            
            # Calculate various engagement metrics
            buying_signals = self.calculate_buying_signals(text, speaker)
            objection_resistance = self.calculate_objection_resistance(text)
            urgency_authority = self.calculate_urgency_authority(text, speaker)
            question_analysis = self.analyze_question_patterns(text, speaker)
            
            # Combine all features
            segment_data = {
                'segment_id': idx,
                'speaker': speaker,
                'message': text,
                **buying_signals,
                **objection_resistance,
                **urgency_authority,
                **question_analysis
            }
            
            engagement_data.append(segment_data)
        
        return pd.DataFrame(engagement_data)
    
    def generate_engagement_summary(self):
        """Generate engagement metrics summary"""
        engagement_df = self.analyze_conversation()
        
        # Overall engagement metrics
        total_buying_signals = engagement_df['total_buying_signals'].sum()
        total_resistance = engagement_df['total_resistance'].sum()
        avg_urgency = engagement_df['urgency_indicators'].mean()
        total_questions = engagement_df['total_questions'].sum()
        
        # Calculate engagement vs resistance ratio
        if total_resistance > 0:
            engagement_ratio = total_buying_signals / total_resistance
        else:
            engagement_ratio = total_buying_signals if total_buying_signals > 0 else 1.0
        
        # By speaker analysis
        speaker_engagement = engagement_df.groupby('speaker').agg({
            'total_buying_signals': 'sum',
            'total_resistance': 'sum',
            'urgency_indicators': 'sum',
            'authority_score': 'mean',
            'question_engagement_score': 'sum'
        }).round(2)
        
        # Authority analysis
        authority_levels = engagement_df['authority_level'].value_counts().to_dict()
        
        summary = {
            'total_buying_signals': total_buying_signals,
            'total_resistance_signals': total_resistance,
            'engagement_resistance_ratio': round(engagement_ratio, 2),
            'average_urgency': round(avg_urgency, 2),
            'total_questions': total_questions,
            'authority_distribution': authority_levels,
            'speaker_engagement': speaker_engagement.to_dict(),
            'engagement_segments': len(engagement_df[engagement_df['total_buying_signals'] > 0])
        }
        
        return summary, engagement_df

# Calculate engagement metrics
engagement_calculator = EngagementMetricsCalculator(df)
engagement_summary, engagement_df = engagement_calculator.generate_engagement_summary()

print("\n🎯 ENGAGEMENT METRICS SUMMARY:")
print(f"🟢 Total buying signals: {engagement_summary['total_buying_signals']}")
print(f"🔴 Total resistance signals: {engagement_summary['total_resistance_signals']}")
print(f"⚖️ Engagement/Resistance ratio: {engagement_summary['engagement_resistance_ratio']}")
print(f"⚡ Average urgency: {engagement_summary['average_urgency']}")
print(f"❓ Total questions: {engagement_summary['total_questions']}")
print(f"👥 Authority distribution: {engagement_summary['authority_distribution']}")
print(f"📊 Engaged segments: {engagement_summary['engagement_segments']}")

# Show top engaged speakers
print("\nTOP ENGAGED SPEAKERS:")
for speaker, signals in engagement_summary['speaker_engagement']['total_buying_signals'].items():
    print(f"{speaker}: {signals} buying signals, {engagement_summary['speaker_engagement']['total_resistance'][speaker]} resistance")


🎯 ENGAGEMENT METRICS SUMMARY:
🟢 Total buying signals: 41
🔴 Total resistance signals: 36
⚖️ Engagement/Resistance ratio: 1.14
⚡ Average urgency: 0.0
❓ Total questions: 29
👥 Authority distribution: {'Low': 67}
📊 Engaged segments: 21

TOP ENGAGED SPEAKERS:
Arjun: 18 buying signals, 12 resistance
Meera: 23 buying signals, 24 resistance
Small talk as both smile and exchange pleasantries before leaving the call.: 0 buying signals, 0 resistance


## 5. Deal Scoring Algorithm

In [38]:
class DealScoringAlgorithm:
    """Advanced deal scoring using multiple quantitative factors"""
    
    def __init__(self, df, financial_df, competitive_df, engagement_df):
        self.df = df
        self.financial_df = financial_df
        self.competitive_df = competitive_df
        self.engagement_df = engagement_df
        
        # Scoring weights (can be tuned based on historical data)
        self.weights = {
            'budget_qualification': 0.25,
            'engagement_score': 0.20,
            'urgency_timeline': 0.15,
            'authority_level': 0.15,
            'competitive_position': 0.10,
            'technical_fit': 0.10,
            'risk_factors': 0.05
        }
    
    def calculate_budget_score(self):
        """Calculate budget qualification score (0-100)"""
        budget_indicators = {
            'currency_mentions': self.financial_df['currency_mentions'].sum(),
            'roi_discussions': self.financial_df['roi_mentions'].sum(),
            'cost_sensitivity': self.financial_df['cost_sensitivity_score'].mean(),
            'approval_complexity': self.financial_df['approval_complexity_score'].mean()
        }
        
        # Budget score calculation
        budget_score = 0
        
        # Currency mentions indicate budget discussions
        if budget_indicators['currency_mentions'] > 0:
            budget_score += min(30, budget_indicators['currency_mentions'] * 10)
        
        # ROI discussions show value focus
        if budget_indicators['roi_discussions'] > 0:
            budget_score += min(25, budget_indicators['roi_discussions'] * 5)
        
        # Lower cost sensitivity is better
        cost_sensitivity_score = max(0, 25 - budget_indicators['cost_sensitivity'] * 5)
        budget_score += cost_sensitivity_score
        
        # Lower approval complexity is better
        approval_score = max(0, 20 - budget_indicators['approval_complexity'] * 5)
        budget_score += approval_score
        
        return min(100, max(0, budget_score))
    
    def calculate_engagement_score(self):
        """Calculate engagement score (0-100)"""
        engagement_indicators = {
            'buying_signals': self.engagement_df['total_buying_signals'].sum(),
            'resistance_signals': self.engagement_df['total_resistance'].sum(),
            'questions': self.engagement_df['total_questions'].sum(),
            'question_engagement': self.engagement_df['question_engagement_score'].sum()
        }
        
        engagement_score = 0
        
        # Buying signals are positive
        engagement_score += min(40, engagement_indicators['buying_signals'] * 5)
        
        # Questions show interest
        engagement_score += min(25, engagement_indicators['questions'] * 3)
        
        # Quality questions are even better
        engagement_score += min(20, engagement_indicators['question_engagement'])
        
        # Resistance signals are negative
        engagement_score -= min(25, engagement_indicators['resistance_signals'] * 3)
        
        return min(100, max(0, engagement_score))
    
    def calculate_urgency_score(self):
        """Calculate urgency and timeline score (0-100)"""
        urgency_indicators = self.engagement_df['urgency_indicators'].sum()
        
        # Look for timeline mentions
        timeline_patterns = [
            r'\d+\s*(?:months?|weeks?|days?)\b',
            r'\b(?:deadline|asap|urgent|immediately|soon)\b',
            r'\b(?:q[1-4]|quarter|fiscal year|end of year)\b'
        ]
        
        timeline_mentions = 0
        all_text = ' '.join(self.df['message'].astype(str).tolist()).lower()
        
        for pattern in timeline_patterns:
            timeline_mentions += len(re.findall(pattern, all_text))
        
        urgency_score = 0
        urgency_score += min(50, urgency_indicators * 10)
        urgency_score += min(50, timeline_mentions * 5)
        
        return min(100, urgency_score)
    
    def calculate_authority_score(self):
        """Calculate decision authority score (0-100)"""
        authority_scores = self.engagement_df['authority_score'].tolist()
        max_authority = max(authority_scores) if authority_scores else 0
        
        # Authority score based on highest authority level found
        if max_authority >= 6:
            return 100
        elif max_authority >= 3:
            return 70
        elif max_authority > 0:
            return 40
        else:
            return 20  # Some authority assumed if in conversation
    
    def calculate_competitive_score(self):
        """Calculate competitive position score (0-100)"""
        competitive_indicators = {
            'competitor_mentions': self.competitive_df['competitor_mentions'].sum(),
            'competitive_sentiment': self.competitive_df['competitive_sentiment_score'].mean(),
            'differentiation_indicators': self.competitive_df['total_differentiation_indicators'].sum()
        }
        
        competitive_score = 50  # Base score
        
        # Positive competitive sentiment is good for us
        if not np.isnan(competitive_indicators['competitive_sentiment']):
            competitive_score += competitive_indicators['competitive_sentiment'] * 25
        
        # Differentiation indicators are positive
        competitive_score += min(30, competitive_indicators['differentiation_indicators'] * 5)
        
        # Too many competitor mentions might indicate strong competition
        if competitive_indicators['competitor_mentions'] > 5:
            competitive_score -= 10
        
        return min(100, max(0, competitive_score))
    
    def calculate_technical_fit_score(self):
        """Calculate technical fit score based on requirements discussion"""
        all_text = ' '.join(self.df['message'].astype(str).tolist()).lower()
        
        # Technical requirement indicators
        technical_patterns = {
            'integration': r'\b(?:integration|integrate|api|connect|sync)\b',
            'scalability': r'\b(?:scale|scalability|growth|volume|capacity)\b',
            'security': r'\b(?:security|secure|compliance|audit|gdpr|soc)\b',
            'architecture': r'\b(?:architecture|infrastructure|cloud|aws|database)\b'
        }
        
        technical_mentions = 0
        for category, pattern in technical_patterns.items():
            mentions = len(re.findall(pattern, all_text))
            technical_mentions += mentions
        
        # More technical discussion usually indicates better fit
        technical_score = min(100, technical_mentions * 5)
        
        # Bonus for specific technical details
        specific_tech = len(re.findall(r'\b(?:node\.js|react|python|kubernetes|docker)\b', all_text))
        technical_score += min(20, specific_tech * 10)
        
        return technical_score
    
    def calculate_risk_factors(self):
        """Calculate risk factors that might hurt the deal (0-100, lower is better)"""
        all_text = ' '.join(self.df['message'].astype(str).tolist()).lower()
        
        risk_indicators = {
            'past_failures': len(re.findall(r'\b(?:past|previous|last time|before|failed|disappointed)\b', all_text)),
            'budget_constraints': len(re.findall(r'\b(?:tight budget|no budget|expensive|can\'t afford)\b', all_text)),
            'timing_issues': len(re.findall(r'\b(?:not ready|too early|later|postpone|delay)\b', all_text)),
            'multiple_vendors': len(re.findall(r'\b(?:other vendors|comparing|evaluation|rfp)\b', all_text))
        }
        
        total_risk = sum(risk_indicators.values())
        
        # Convert to risk score (lower is better)
        risk_score = max(0, 100 - total_risk * 10)
        
        return risk_score
    
    def calculate_overall_deal_score(self):
        """Calculate overall deal score using weighted factors"""
        scores = {
            'budget_qualification': self.calculate_budget_score(),
            'engagement_score': self.calculate_engagement_score(),
            'urgency_timeline': self.calculate_urgency_score(),
            'authority_level': self.calculate_authority_score(),
            'competitive_position': self.calculate_competitive_score(),
            'technical_fit': self.calculate_technical_fit_score(),
            'risk_factors': self.calculate_risk_factors()
        }
        
        # Calculate weighted overall score
        overall_score = sum(scores[factor] * weight for factor, weight in self.weights.items())
        
        # Determine deal grade
        if overall_score >= 80:
            grade = 'A - Hot Lead'
        elif overall_score >= 65:
            grade = 'B - Qualified'
        elif overall_score >= 50:
            grade = 'C - Needs Nurturing'
        elif overall_score >= 35:
            grade = 'D - Long Shot'
        else:
            grade = 'F - Unlikely'
        
        # Calculate win probability based on score
        win_probability = min(95, max(5, overall_score * 1.2 - 10))
        
        return {
            'individual_scores': scores,
            'overall_score': round(overall_score, 1),
            'deal_grade': grade,
            'win_probability': round(win_probability, 1),
            'score_factors': {factor: round(score * weight, 1) 
                           for factor, score, weight in 
                           [(f, scores[f], self.weights[f]) for f in scores.keys()]}
        }

# Calculate deal score
deal_scorer = DealScoringAlgorithm(df, financial_df, competitive_df, engagement_df)
deal_score_results = deal_scorer.calculate_overall_deal_score()

print("\n🎯 DEAL SCORING ANALYSIS:")
print(f"📊 Overall Score: {deal_score_results['overall_score']}/100")
print(f"🏆 Deal Grade: {deal_score_results['deal_grade']}")
print(f"🎲 Win Probability: {deal_score_results['win_probability']}%")

print("\nSCORE BREAKDOWN:")
for factor, score in deal_score_results['individual_scores'].items():
    weighted_score = deal_score_results['score_factors'][factor]
    print(f"{factor.replace('_', ' ').title()}: {score:.1f}/100 (weighted: {weighted_score:.1f})")

print("\nKEY INSIGHTS:")
top_factors = sorted(deal_score_results['individual_scores'].items(), key=lambda x: x[1], reverse=True)[:3]
print(f"✅ Strengths: {', '.join([f.replace('_', ' ') for f, _ in top_factors])}")

weak_factors = sorted(deal_score_results['individual_scores'].items(), key=lambda x: x[1])[:2]
print(f"⚠️ Areas for improvement: {', '.join([f.replace('_', ' ') for f, _ in weak_factors])}")


🎯 DEAL SCORING ANALYSIS:
📊 Overall Score: 59.7/100
🏆 Deal Grade: C - Needs Nurturing
🎲 Win Probability: 61.7%

SCORE BREAKDOWN:
Budget Qualification: 83.9/100 (weighted: 21.0)
Engagement Score: 60.0/100 (weighted: 12.0)
Urgency Timeline: 15.0/100 (weighted: 2.2)
Authority Level: 20.0/100 (weighted: 3.0)
Competitive Position: 70.0/100 (weighted: 7.0)
Technical Fit: 120.0/100 (weighted: 12.0)
Risk Factors: 50.0/100 (weighted: 2.5)

KEY INSIGHTS:
✅ Strengths: technical fit, budget qualification, competitive position
⚠️ Areas for improvement: urgency timeline, authority level


## 6. Enhanced Business Intelligence Dashboard

In [39]:
class BusinessIntelligenceDashboard:
    """Create comprehensive business intelligence visualizations"""
    
    def __init__(self, df, financial_summary, competitive_summary, engagement_summary, deal_score_results):
        self.df = df
        self.financial_summary = financial_summary
        self.competitive_summary = competitive_summary
        self.engagement_summary = engagement_summary
        self.deal_score_results = deal_score_results
    
    def create_deal_score_dashboard(self):
        """Create deal scoring dashboard"""
        # Deal score radar chart
        categories = list(self.deal_score_results['individual_scores'].keys())
        scores = list(self.deal_score_results['individual_scores'].values())
        
        fig = go.Figure()
        
        fig.add_trace(go.Scatterpolar(
            r=scores,
            theta=[cat.replace('_', ' ').title() for cat in categories],
            fill='toself',
            name='Current Deal',
            line_color='rgb(255, 99, 71)'
        ))
        
        # Add benchmark line at 70 (good score)
        fig.add_trace(go.Scatterpolar(
            r=[70] * len(categories),
            theta=[cat.replace('_', ' ').title() for cat in categories],
            fill='toself',
            name='Target Benchmark',
            line_color='rgb(100, 200, 100)',
            opacity=0.3
        ))
        
        fig.update_layout(
            polar=dict(
                radialaxis=dict(
                    visible=True,
                    range=[0, 100]
                )
            ),
            showlegend=True,
            title=f"Deal Score Analysis: {self.deal_score_results['deal_grade']} ({self.deal_score_results['overall_score']}/100)"
        )
        
        return fig
    
    def create_sentiment_timeline(self):
        """Create sentiment progression over conversation"""
        if 'sentiment_compound' in self.df.columns:
            fig = make_subplots(
                rows=2, cols=1,
                subplot_titles=('Sentiment Over Time', 'Engagement Signals'),
                row_heights=[0.6, 0.4]
            )
            
            # Sentiment timeline
            for speaker in self.df['speaker'].unique():
                speaker_df = self.df[self.df['speaker'] == speaker]
                fig.add_trace(
                    go.Scatter(
                        x=speaker_df['start_seconds'],
                        y=speaker_df['sentiment_compound'],
                        mode='lines+markers',
                        name=f'{speaker} Sentiment',
                        line=dict(width=2)
                    ),
                    row=1, col=1
                )
            
            # Add neutral line
            fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=1)
            
            # Question frequency
            fig.add_trace(
                go.Bar(
                    x=self.df['start_seconds'],
                    y=self.df['question_marks'],
                    name='Questions',
                    marker_color='lightblue',
                    opacity=0.7
                ),
                row=2, col=1
            )
            
            fig.update_xaxes(title_text="Time (seconds)", row=2, col=1)
            fig.update_yaxes(title_text="Sentiment", range=[-1, 1], row=1, col=1)
            fig.update_yaxes(title_text="Questions", row=2, col=1)
            
            fig.update_layout(
                height=600,
                title="Conversation Sentiment & Engagement Timeline"
            )
            
            return fig
        return None
    
    def create_business_intelligence_summary(self):
        """Create comprehensive business intelligence summary"""
        # Create metrics summary table
        summary_data = {
            'Category': ['Financial', 'Competitive', 'Engagement', 'Authority', 'Technical', 'Overall'],
            'Key Metric': [
                f"${self.financial_summary['estimated_deal_size']:,.0f} potential" if self.financial_summary['estimated_deal_size'] > 0 else "No specific amount",
                f"{self.competitive_summary['unique_competitors']} competitors mentioned",
                f"{self.engagement_summary['engagement_resistance_ratio']} engagement ratio",
                f"{list(self.engagement_summary['authority_distribution'].keys())[0] if self.engagement_summary['authority_distribution'] else 'Unknown'} authority",
                f"{self.deal_score_results['individual_scores']['technical_fit']:.0f}/100 fit score",
                f"{self.deal_score_results['win_probability']:.0f}% win probability"
            ],
            'Status': [
                '🟢 Qualified' if self.financial_summary['total_currency_mentions'] > 0 else '🟡 Unqualified',
                '🟢 Favorable' if self.competitive_summary['average_competitive_sentiment'] > 0 else '🔴 Challenging',
                '🟢 High' if self.engagement_summary['engagement_resistance_ratio'] > 2 else '🟡 Medium' if self.engagement_summary['engagement_resistance_ratio'] > 1 else '🔴 Low',
                '🟢 High' if 'High' in str(self.engagement_summary['authority_distribution']) else '🟡 Medium',
                '🟢 Strong' if self.deal_score_results['individual_scores']['technical_fit'] > 70 else '🟡 Moderate',
                self.deal_score_results['deal_grade'].split(' - ')[0]
            ]
        }
        
        summary_df = pd.DataFrame(summary_data)
        
        fig = go.Figure(data=[go.Table(
            header=dict(values=list(summary_df.columns),
                       fill_color='lightblue',
                       align='left',
                       font=dict(size=14, color='black')),
            cells=dict(values=[summary_df[col] for col in summary_df.columns],
                      fill_color='white',
                      align='left',
                      font=dict(size=12),
                      height=40))
        ])
        
        fig.update_layout(
            title="Business Intelligence Summary Dashboard",
            height=400
        )
        
        return fig
    
    def create_competitive_positioning_chart(self):
        """Create competitive positioning visualization"""
        if self.competitive_summary['total_competitor_mentions'] > 0:
            competitors = list(self.competitive_summary['top_competitors'].keys())[:5]
            mentions = list(self.competitive_summary['top_competitors'].values())[:5]
            
            fig = px.bar(
                x=mentions,
                y=competitors,
                orientation='h',
                title='Competitive Landscape - Competitor Mentions',
                labels={'x': 'Number of Mentions', 'y': 'Competitors'},
                color=mentions,
                color_continuous_scale='Reds'
            )
            
            fig.update_layout(
                height=400,
                showlegend=False,
                annotations=[
                    dict(
                        text=f"Competitive Sentiment: {self.competitive_summary['average_competitive_sentiment']:.2f} (Opportunity Score)",
                        showarrow=False,
                        xref="paper", yref="paper",
                        x=0.5, y=1.1, xanchor='center',
                        font=dict(size=12)
                    )
                ]
            )
            
            return fig
        return None
    
    def create_financial_intelligence_gauge(self):
        """Create financial qualification gauge"""
        budget_score = self.deal_score_results['individual_scores']['budget_qualification']
        
        fig = go.Figure(go.Indicator(
            mode = "gauge+number+delta",
            value = budget_score,
            domain = {'x': [0, 1], 'y': [0, 1]},
            title = {'text': "Budget Qualification Score"},
            delta = {'reference': 70},
            gauge = {
                'axis': {'range': [None, 100]},
                'bar': {'color': "darkblue"},
                'steps': [
                    {'range': [0, 25], 'color': "lightgray"},
                    {'range': [25, 50], 'color': "gray"},
                    {'range': [50, 75], 'color': "lightgreen"},
                    {'range': [75, 100], 'color': "green"}
                ],
                'threshold': {
                    'line': {'color': "red", 'width': 4},
                    'thickness': 0.75,
                    'value': 70
                }
            }
        ))
        
        fig.update_layout(
            height=400,
            title="Financial Intelligence Dashboard",
            annotations=[
                dict(
                    text=f"Currency Mentions: {self.financial_summary['total_currency_mentions']}<br>ROI Discussions: {self.financial_summary['roi_discussion_segments']}",
                    showarrow=False,
                    x=0.5, y=0.2,
                    font=dict(size=12)
                )
            ]
        )
        
        return fig

# Create enhanced dashboard
dashboard = BusinessIntelligenceDashboard(
    df, financial_summary, competitive_summary, engagement_summary, deal_score_results
)

# Display all visualizations
print("📊 BUSINESS INTELLIGENCE DASHBOARD")
print("==" * 25)

dashboard.create_business_intelligence_summary().write_html("plot21.html")
dashboard.create_deal_score_dashboard().write_html("plot22.html")
dashboard.create_financial_intelligence_gauge().write_html("plot23.html")

if dashboard.create_competitive_positioning_chart():
    dashboard.create_competitive_positioning_chart().write_html("plot24.html")
    
if dashboard.create_sentiment_timeline():
    dashboard.create_sentiment_timeline().write_html("plot25.html")

print("\n✅ Enhanced Business Intelligence Analysis Complete!")
print(f"📈 Overall Deal Score: {deal_score_results['overall_score']}/100 ({deal_score_results['deal_grade']})")
print(f"🎯 Win Probability: {deal_score_results['win_probability']}%")
print(f"💰 Financial Intelligence: {financial_summary['total_currency_mentions']} currency mentions")
print(f"🏢 Competitive Intelligence: {competitive_summary['total_competitor_mentions']} competitor mentions")
print(f"🎪 Engagement Intelligence: {engagement_summary['total_buying_signals']} buying signals")

📊 BUSINESS INTELLIGENCE DASHBOARD

✅ Enhanced Business Intelligence Analysis Complete!
📈 Overall Deal Score: 59.7/100 (C - Needs Nurturing)
🎯 Win Probability: 61.7%
💰 Financial Intelligence: 3 currency mentions
🏢 Competitive Intelligence: 11 competitor mentions
🎪 Engagement Intelligence: 41 buying signals


## 7. Automated Reporting & Export System

In [40]:
class AutomatedReportGenerator:
    """Generate comprehensive automated reports with NLP insights"""
    
    def __init__(self, df, financial_summary, competitive_summary, engagement_summary, deal_score_results):
        self.df = df
        self.financial_summary = financial_summary
        self.competitive_summary = competitive_summary
        self.engagement_summary = engagement_summary
        self.deal_score_results = deal_score_results
        self.conversation_date = datetime.now().strftime("%Y-%m-%d")
    
    def generate_executive_summary(self):
        """Generate executive summary with key insights"""
        duration_minutes = self.df['end_seconds'].max() / 60
        
        summary = f"""
# SALES INTELLIGENCE REPORT
**Date:** {self.conversation_date}
**Duration:** {duration_minutes:.1f} minutes
**Participants:** {', '.join(self.df['speaker'].unique())}

## 🎯 EXECUTIVE SUMMARY
**Deal Score:** {self.deal_score_results['overall_score']}/100 ({self.deal_score_results['deal_grade']})
**Win Probability:** {self.deal_score_results['win_probability']}%
**Recommendation:** {self.generate_recommendation()}

## 💰 FINANCIAL INTELLIGENCE
- **Budget Discussions:** {self.financial_summary['total_currency_mentions']} currency mentions
- **Deal Size Estimate:** ${self.financial_summary['estimated_deal_size']:,.0f}" if self.financial_summary['estimated_deal_size'] > 0 else "Not specified"
- **Cost Sensitivity:** {self.financial_summary['average_cost_sensitivity']:.1f}/10
- **ROI Focus:** {self.financial_summary['roi_discussion_segments']} segments

## 🏢 COMPETITIVE LANDSCAPE
- **Competitors Mentioned:** {', '.join(list(self.competitive_summary['top_competitors'].keys())[:3]) if self.competitive_summary['top_competitors'] else 'None'}
- **Competitive Sentiment:** {self.competitive_summary['average_competitive_sentiment']:.2f} (opportunity score)
- **Primary Differentiation:** {self.competitive_summary['primary_differentiation_strategy']}

## 🎪 ENGAGEMENT ANALYSIS
- **Buying Signals:** {self.engagement_summary['total_buying_signals']} positive indicators
- **Resistance Signals:** {self.engagement_summary['total_resistance_signals']} concerns
- **Engagement Ratio:** {self.engagement_summary['engagement_resistance_ratio']} (signals vs resistance)
- **Authority Level:** {list(self.engagement_summary['authority_distribution'].keys())[0] if self.engagement_summary['authority_distribution'] else 'Unknown'}

## 📊 KEY PERFORMANCE INDICATORS
""" + self.generate_kpi_section() + f"""

## 🎯 NEXT STEPS & RECOMMENDATIONS
""" + self.generate_next_steps() + f"""

## ⚠️ RISK FACTORS
""" + self.generate_risk_assessment()
        
        return summary
    
    def generate_recommendation(self):
        """Generate action recommendation based on score"""
        score = self.deal_score_results['overall_score']
        
        if score >= 80:
            return "🟢 HIGH PRIORITY - Fast-track this opportunity with immediate follow-up"
        elif score >= 65:
            return "🟡 QUALIFIED - Continue active pursuit with structured follow-up"
        elif score >= 50:
            return "🟠 NURTURE - Develop relationship with educational content and patience"
        else:
            return "🔴 RECONSIDER - Evaluate if resources are better allocated elsewhere"
    
    def generate_kpi_section(self):
        """Generate KPI analysis section"""
        scores = self.deal_score_results['individual_scores']
        
        kpi_analysis = ""
        
        # Identify top 2 strengths and weaknesses
        strengths = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:2]
        weaknesses = sorted(scores.items(), key=lambda x: x[1])[:2]
        
        kpi_analysis += "### 🚀 Top Strengths:\n"
        for strength, score in strengths:
            kpi_analysis += f"- **{strength.replace('_', ' ').title()}:** {score:.1f}/100\n"
        
        kpi_analysis += "\n### ⚠️ Areas for Improvement:\n"
        for weakness, score in weaknesses:
            kpi_analysis += f"- **{weakness.replace('_', ' ').title()}:** {score:.1f}/100\n"
        
        return kpi_analysis
    
    def generate_next_steps(self):
        """Generate personalized next steps based on analysis"""
        steps = []
        
        # Financial next steps
        if self.financial_summary['total_currency_mentions'] == 0:
            steps.append("💰 **Budget Discovery:** Schedule follow-up to understand budget parameters and approval process")
        
        # Competitive next steps
        if self.competitive_summary['total_competitor_mentions'] > 0:
            steps.append(f"🏢 **Competitive Strategy:** Prepare differentiation materials against {', '.join(list(self.competitive_summary['top_competitors'].keys())[:2])}")
        
        # Engagement next steps
        if self.engagement_summary['engagement_resistance_ratio'] < 2:
            steps.append("🎪 **Objection Handling:** Address concerns and provide additional proof points")
        
        # Technical next steps
        if self.deal_score_results['individual_scores']['technical_fit'] < 70:
            steps.append("🔧 **Technical Deep Dive:** Schedule technical discovery session with their engineering team")
        
        # Authority next steps
        if 'Low' in str(self.engagement_summary['authority_distribution']):
            steps.append("👥 **Stakeholder Mapping:** Identify and engage decision makers and influencers")
        
        # Urgency next steps
        if self.deal_score_results['individual_scores']['urgency_timeline'] < 50:
            steps.append("⏰ **Create Urgency:** Develop business case showing cost of delayed decision")
        
        return "\n".join(steps) if steps else "✅ **Continue Current Strategy:** Maintain momentum with regular follow-ups"
    
    def generate_risk_assessment(self):
        """Generate risk assessment based on conversation"""
        risks = []
        
        if self.engagement_summary['total_resistance_signals'] > self.engagement_summary['total_buying_signals']:
            risks.append("🔴 **High Resistance:** More objections than buying signals detected")
        
        if self.competitive_summary['average_competitive_sentiment'] < -0.3:
            risks.append("🔴 **Strong Competition:** Competitors viewed favorably by prospect")
        
        if self.financial_summary['average_cost_sensitivity'] > 7:
            risks.append("🟡 **Price Sensitivity:** High concern about costs and budget constraints")
        
        if self.deal_score_results['individual_scores']['authority_level'] < 50:
            risks.append("🟡 **Authority Gap:** May not be engaging with final decision maker")
        
        if len(self.competitive_summary['top_competitors']) > 3:
            risks.append("🟡 **Crowded Evaluation:** Multiple competitors in consideration")
        
        return "\n".join(risks) if risks else "✅ **Low Risk:** No significant risk factors identified"
    
    def export_comprehensive_report(self, filename_prefix='sales_intelligence_report'):
        """Export comprehensive analysis to multiple formats"""
        
        # 1. Executive Summary (Markdown)
        executive_summary = self.generate_executive_summary()
        with open(f'{filename_prefix}_executive_summary.md', 'w', encoding='utf-8') as f:
            f.write(executive_summary)
        
        # 2. Detailed Financial Analysis (CSV)
        financial_details = {
            'Metric': [
                'Total Currency Mentions', 'Estimated Deal Size', 'Average Cost Sensitivity',
                'ROI Discussion Segments', 'Budget Qualification Score'
            ],
            'Value': [
                self.financial_summary['total_currency_mentions'],
                self.financial_summary['estimated_deal_size'],
                round(self.financial_summary['average_cost_sensitivity'], 2),
                self.financial_summary['roi_discussion_segments'],
                round(self.deal_score_results['individual_scores']['budget_qualification'], 1)
            ]
        }
        pd.DataFrame(financial_details).to_csv(f'{filename_prefix}_financial_analysis.csv', index=False)
        
        # 3. Competitive Intelligence (CSV)
        competitive_data = []
        for competitor, mentions in self.competitive_summary['top_competitors'].items():
            competitive_data.append({
                'Competitor': competitor,
                'Mentions': mentions,
                'Sentiment': self.competitive_summary['average_competitive_sentiment']
            })
        
        if competitive_data:
            pd.DataFrame(competitive_data).to_csv(f'{filename_prefix}_competitive_analysis.csv', index=False)
        
        # 4. Deal Scoring Details (CSV)
        scoring_data = {
            'Scoring_Factor': list(self.deal_score_results['individual_scores'].keys()),
            'Score': list(self.deal_score_results['individual_scores'].values()),
            'Weighted_Score': list(self.deal_score_results['score_factors'].values())
        }
        pd.DataFrame(scoring_data).to_csv(f'{filename_prefix}_deal_scoring.csv', index=False)
        
        # 5. Master Summary (CSV)
        master_summary = {
            'Conversation_Date': [self.conversation_date],
            'Duration_Minutes': [round(self.df['end_seconds'].max() / 60, 1)],
            'Participants': [', '.join(self.df['speaker'].unique())],
            'Overall_Score': [self.deal_score_results['overall_score']],
            'Deal_Grade': [self.deal_score_results['deal_grade']],
            'Win_Probability': [self.deal_score_results['win_probability']],
            'Budget_Mentions': [self.financial_summary['total_currency_mentions']],
            'Competitor_Mentions': [self.competitive_summary['total_competitor_mentions']],
            'Buying_Signals': [self.engagement_summary['total_buying_signals']],
            'Resistance_Signals': [self.engagement_summary['total_resistance_signals']]
        }
        pd.DataFrame(master_summary).to_csv(f'{filename_prefix}_master_summary.csv', index=False)
        
        return {
            'executive_summary': f'{filename_prefix}_executive_summary.md',
            'financial_analysis': f'{filename_prefix}_financial_analysis.csv',
            'competitive_analysis': f'{filename_prefix}_competitive_analysis.csv',
            'deal_scoring': f'{filename_prefix}_deal_scoring.csv',
            'master_summary': f'{filename_prefix}_master_summary.csv'
        }

# Generate comprehensive report
report_generator = AutomatedReportGenerator(
    df, financial_summary, competitive_summary, engagement_summary, deal_score_results
)

# Export all reports
exported_files = report_generator.export_comprehensive_report()

# Display executive summary
executive_summary = report_generator.generate_executive_summary()
print(executive_summary)

print("\n" + "=" * 50)
print("📄 EXPORTED FILES:")
for report_type, filename in exported_files.items():
    print(f"📊 {report_type.replace('_', ' ').title()}: {filename}")

print(f"\n✅ COMPREHENSIVE SALES INTELLIGENCE ANALYSIS COMPLETE!")
print(f"🎯 Overall Assessment: {deal_score_results['deal_grade']}")
print(f"📈 Win Probability: {deal_score_results['win_probability']}%")
print(f"🚀 Recommendation: {report_generator.generate_recommendation()}")


# SALES INTELLIGENCE REPORT
**Date:** 2025-08-24
**Duration:** 25.3 minutes
**Participants:** Arjun, Meera, Small talk as both smile and exchange pleasantries before leaving the call.

## 🎯 EXECUTIVE SUMMARY
**Deal Score:** 59.7/100 (C - Needs Nurturing)
**Win Probability:** 61.7%
**Recommendation:** 🟠 NURTURE - Develop relationship with educational content and patience

## 💰 FINANCIAL INTELLIGENCE
- **Budget Discussions:** 3 currency mentions
- **Deal Size Estimate:** $100,000" if self.financial_summary['estimated_deal_size'] > 0 else "Not specified"
- **Cost Sensitivity:** 0.1/10
- **ROI Focus:** 2 segments

## 🏢 COMPETITIVE LANDSCAPE
- **Competitors Mentioned:** Project44, AWS, competitors
- **Competitive Sentiment:** 0.00 (opportunity score)
- **Primary Differentiation:** none

## 🎪 ENGAGEMENT ANALYSIS
- **Buying Signals:** 41 positive indicators
- **Resistance Signals:** 36 concerns
- **Engagement Ratio:** 1.14 (signals vs resistance)
- **Authority Level:** Low

## 📊 KEY PERFORMA

In [42]:
# How to make cookies and run them in a jar